In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

In [21]:
SEED=42
torch.manual_seed(SEED)
np.random.seed(SEED)

Preparing Data

In [22]:
data=pd.read_csv('/content/MicrosoftStock.csv',on_bad_lines='skip')
data['date']=pd.to_datetime(data['date'])
data=data.sort_values('date')
data.head()

,index,date,open,high,low,close,volume,Name
0,390198,2013-02-08,27.35,27.71,27.31,27.55,33318306,MSFT
1,390199,2013-02-11,27.65,27.92,27.50,27.86,32247549,MSFT
2,390200,2013-02-12,27.88,28.00,27.75,27.88,35990829,MSFT
3,390201,2013-02-13,27.93,28.11,27.88,28.03,41715530,MSFT
4,390202,2013-02-14,27.92,28.06,27.87,28.04,32663174,MSFT


In [23]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   index   1259 non-null   int64         
 1   date    1259 non-null   datetime64[ns]
 2   open    1259 non-null   float64       
 3   high    1259 non-null   float64       
 4   low     1259 non-null   float64       
 5   close   1259 non-null   float64       
 6   volume  1259 non-null   int64         
 7   Name    1259 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(2), object(1)
memory usage: 78.8+ KB


In [24]:
#Since we are doing multi-variate analysis we take multiple variables into consideration
features=data[['open','high','low','close','volume']]

In [25]:
train_size=int(0.8*len(features))
val_size=int(0.1*len(features))
train_data=features[:train_size]
val_data=features[train_size:train_size+val_size]
test_data=features[train_size+val_size:]

In [26]:
train_data.shape,val_data.shape,test_data.shape,

((1007, 5), (125, 5), (127, 5))

Scaling the data

In [27]:
scaler=RobustScaler()
train_data=scaler.fit_transform(train_data)
val_data=scaler.transform(val_data)
test_data=scaler.transform(test_data)

Sequencing

In [28]:
def create_seq(data,seq_len=40):
  xs,ys=[],[]
  for i in range(seq_len,len(data)):
    xs.append(data[i-seq_len:i,:])
    ys.append(data[i,3])
  return np.array(xs),np.array(ys)

In [29]:
x_train,y_train=create_seq(train_data)
x_val,y_val=create_seq(val_data)
x_test,y_test=create_seq(test_data)

In [30]:
x_train.shape,y_train.shape,x_test.shape,y_test.shape,x_val.shape,y_val.shape

((967, 40, 5), (967,), (87, 40, 5), (87,), (85, 40, 5), (85,))

In [31]:
x_train=torch.tensor(x_train).float()
y_train=torch.tensor(y_train).float()
x_test=torch.tensor(x_test).float()
y_test=torch.tensor(y_test).float()
x_val=torch.tensor(x_val).float()
y_val=torch.tensor(y_val).float()
train_loader=DataLoader(TensorDataset(x_train,y_train),batch_size=16,shuffle=True)
test_loader=DataLoader(TensorDataset(x_test,y_test),batch_size=16,shuffle=True)
val_loader=DataLoader(TensorDataset(x_val,y_val),batch_size=16,shuffle=True)

In [42]:
for xb, yb in train_loader:
  print(f"Type of input features (xb): {type(xb)}")
  print(f"Shape of input features (xb): {xb.shape}")
  print(f"Type of target values (yb): {type(yb)}")
  print(f"Shape of target values (yb): {yb.shape}")
  break

Type of input features (xb): <class 'torch.Tensor'>
Shape of input features (xb): torch.Size([16, 40, 5])
Type of target values (yb): <class 'torch.Tensor'>
Shape of target values (yb): torch.Size([16])


Model

In [33]:
class MSFTStockLSTM(nn.Module):
  def __init__(self):
    super().__init__()
    self.lstm1=nn.LSTM(input_size=5,hidden_size=64,batch_first=True) # Updated input_size to 5
    self.lstm2=nn.LSTM(input_size=64,hidden_size=64,batch_first=True)
    self.fc=nn.Sequential(
        nn.Linear(in_features=64,out_features=32),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(in_features=32,out_features=1)
    )
  def forward(self,x):
    x,_=self.lstm1(x)
    x,hidden_state=self.lstm2(x)
    out=x[:,-1,:]
    return self.fc(out)

In [34]:
model=MSFTStockLSTM()
criterion=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [43]:
epochs=15
for epoch in range(epochs):
  model.train()
  for xb,yb in train_loader:
    optimizer.zero_grad()
    preds=model(xb)
    loss=criterion(preds, yb.unsqueeze(1))
    loss.backward()
    optimizer.step()

model.eval()
val_loss=[]
with torch.inference_mode():
  for xb,yb in val_loader:
    vpreds=model(xb)
    val_loss.append(criterion(vpreds, yb.unsqueeze(1)))
print(f"Epoch {epoch+1}/{epochs}, Val MSE={np.mean(val_loss):.6f}")

Epoch 15/15, Val MSE=0.200581


In [44]:
#testing
model.eval()
preds_list=[]
with torch.inference_mode():
  for xb,yb in test_loader:
    preds_list.append(model(xb))

preds = torch.cat(preds_list).cpu().numpy() # Concatenate all predictions and convert to numpy

# To inverse transform, we need to create a dummy array with 5 features
# and place the 1-feature predictions/actuals into the correct column (index 3 for 'close')
num_features = features.shape[1] # This is 5
dummy_preds = np.zeros((preds.shape[0], num_features))
dummy_y_test = np.zeros((y_test.shape[0], num_features))

dummy_preds[:, 3] = preds.flatten() # Place predictions in the 'close' column
dummy_y_test[:, 3] = y_test.numpy() # Place true values in the 'close' column

preds_unscaled = scaler.inverse_transform(dummy_preds)[:, 3] # Inverse transform and extract 'close'
y_test_unscaled = scaler.inverse_transform(dummy_y_test)[:, 3] # Inverse transform and extract 'close'

mse=np.mean((preds_unscaled - y_test_unscaled)**2)
rmse=np.sqrt(mse)

print(f"MSE:{mse:.6f}")
print(f"RMSE:{rmse:.6f}")

MSE:284.170295
RMSE:16.857351
